# Human Readable

The goal of the code is to take a Spotify playlist and sort the songs based on human-readable descriptor, such as danceability, energy, speechiness, acousticness, liveness, instrumentalness, valence... (high-level audio features).

# Preliminary actions

In order to understand how to use "Spotify for Developers" and the "Audio Feature API", let's analyse the songs' high-level features.

## Get Client_ID and Client Secret:

1. Go to your Dashboard: https://developer.spotify.com/dashboard
2. Create a new app: you can insert assign any value to each form fields
3. Click on the name of the app you have just created (My App)
4. Click on the Settings button
5. get your Client_ID and Client Secret

In [1]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

client_id = "41604f5c8fae44a28324c74dd8a2b1fa"
client_secret = "4e0b5d86576b46beada5f035b3b7e95e"

sp = spotipy.Spotify(auth_manager = SpotifyClientCredentials(client_id, client_secret))

## Song search

In [2]:
results = sp.search(q = 'stand by me', limit = 20)

for idx, track in enumerate(results['tracks']['items']):
    print(idx, track['artists'][0]['name'])

0 Ben E. King
1 Lil Durk
2 Ben E. King
3 Prince Royce
4 Ben E. King
5 Otis Redding
6 Young the Giant
7 Florence + The Machine
8 Young the Giant
9 Oasis
10 Oasis
11 Tracy Chapman
12 Journey
13 Mickey Gilley
14 Yung Gravy
15 John Lennon
16 Skylar Grey
17 Fridayy
18 The Clash
19 Dave Fenley


In [3]:
# Taking two results from previous search
res_1 = results['tracks']['items'][5]
res_2 = results['tracks']['items'][11]

songs_title = [res_1['name'], res_2['name']] 
songs_artist = [res_1['artists'][0]['name'], res_2['artists'][0]['name']] 
songs_id = [res_1['id'], res_2['id']]

print(songs_id)

['1aj4GXfmEYXfdVZohCpNKu', '2gs8HVC6KXOQe76XggzZH5']


## Audio feature APIs

In [10]:
modes = ["minor", "major"]
key_tonal = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

audio_features = sp.audio_features(tracks = songs_id)

for i in range(0,2):
    print("%s by %s" % (songs_title[i], songs_artist[i]))
    print("Duration: %.3f seconds" % (audio_features[i]["duration_ms"]/1000))
    print("BPM: %d" % audio_features[i]["tempo"])
    print("Key: %s-%s" % (key_tonal[audio_features[i]["key"]], modes[audio_features[i]["mode"]]))

    for feature in ["danceability", "energy", "speechiness", "acousticness",
                    "liveness", "instrumentalness", "valence"]:
        print("The %s of the song is %1.f %%" % (feature, 100*audio_features[i][feature]))
    print()

Stand by Me by Otis Redding
Duration: 172.333 seconds
BPM: 111
Key: A#-major
The danceability of the song is 88 %
The energy of the song is 46 %
The speechiness of the song is 6 %
The acousticness of the song is 28 %
The liveness of the song is 7 %
The instrumentalness of the song is 0 %
The valence of the song is 92 %

Stand by Me - Live at the Late Show with David Letterman by Tracy Chapman
Duration: 169.013 seconds
BPM: 99
Key: G-major
The danceability of the song is 82 %
The energy of the song is 14 %
The speechiness of the song is 4 %
The acousticness of the song is 80 %
The liveness of the song is 48 %
The instrumentalness of the song is 0 %
The valence of the song is 52 %



## Main code

Let's know generate a personal playlist and sort the songs with a specific danceability curve (not simply ascending).

In [ ]:
import os
import json
import numpy as np
os.chdir(os.path.abspath(os.path.dirname(__file__)))
import spotipy
from spotipy.oauth2 import SpotifyOAuth

* Go to https://open.spotify.com/ , top right corner, press "Account"
* Look at your username

In [ ]:
client_id = "41604f5c8fae44a28324c74dd8a2b1fa"
client_secret = "4e0b5d86576b46beada5f035b3b7e95e"
redirect_uri = "https://example.org/callback"
username = "31tovx3qvk2fk3krr3dc4ripby3q"

scope = 'playlist-modify-public, playlist-modify-private'
sp = spotipy.Spotify(auth_manager = SpotifyOAuth(client_id = client_id,
                                                client_secret = client_secret,
                                                redirect_uri = redirect_uri,
                                                scope = scope))

### Get the list of songs

In [ ]:
assert os.path.exists("list_of_songs.json"), "Please put here a list of songs in list_of_songs.json"

with open("list_of_songs.json",'r') as fp:
    ids = json.load(fp)["ids"]

### Get the audio features

In [ ]:
audio_features = sp.audio_features(tracks = ids)

### Sort the songs

Function to sort the songs based on the danceability (not simply ascending):

In [ ]:
def sort_songs(audio_features, TEACHER_CODE):
    """"Receive audio features and sort them according to your criterion"

    Args:
        audio_features (list of dictionaries): List of songs with audio features

    Returns:
        array of sorted id songs
    """

    sorted_ids = []
    
    if TEACHER_CODE:
        random_idxs=np.random.permutation(len(audio_features))
        for idx in random_idxs:
            sorted_ids.append(audio_features[idx]['id'])
    else:
        danceability = np.array([af["danceability"] for af in audio_features])  # array of danceability values
        idxs = np.argsort(danceability)     # ascendent order

        N_third = int(len(audio_features)/3)    # n_songs / 3
        low_d = idxs[0:N_third]             # low danceability indexes
        mid_d = idxs[N_third, 2*N_third]    # middle danceability indexes
        high_d = idxs[2*N_third, 3*N_third] # high danceability indexes
        
        sorted_idxs = np.concatenate([mid_d, high_d, low_d[::-1]])   # concatenation, low_d in reverse order
        for idx in sorted_idxs:
            sorted_ids.append(audio_features[idx]['id'])    # song ids

    return sorted_ids

In [ ]:
shuffled_songs = sort_songs(audio_features, True)   # flag for TEACHER_CODE

### Create the playlist

In [ ]:
playlist_name = 'CPAC party 2024'
playlist_description = 'Created during CPAC'
playlist = sp.user_playlist_create(username, playlist_name, public = True, collaborative = False,
                                   description = playlist_description)

### Fill the playlist

In [ ]:
results = sp.playlist_add_items(playlist['id'], shuffled_songs, position = None)